<!-- Data_Exploration.ipynb -->

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

print("Libraries imported successfully")
print("Pandas version:", pd.__version__)
print("GeoPandas version:", gpd.__version__)

Libraries imported successfully
Pandas version: 3.0.3
GeoPandas version: 1.1.3


In [2]:
DATA_RAW = 'Data/raw'

crime_files = {
    2022: DATA_RAW + '/Crime data 2022.csv',
    2023: DATA_RAW + '/Crime data 2023.csv',
    2024: DATA_RAW + '/Crime data 2024.csv',
    2025: DATA_RAW + '/Crime data 2025.csv',
    2026: DATA_RAW + '/Crime data 2026.csv',
}

street_poles_file = DATA_RAW + '/Street_Poles.csv'
police_stations_file = DATA_RAW + '/Police_Stations.csv'
intersection_controls_file = DATA_RAW + '/Intersection_Controls.geojson'
weather_file = DATA_RAW + '/Weather.csv'

hospitals_file = DATA_RAW + '/pa_hospitals.geojson'
census_folder = DATA_RAW + '/Census_Tracts_2010-shp'
police_districts_folder = DATA_RAW + '/police_districts'

print("File paths defined")
print("Crime files:", len(crime_files), "years")
print("Street poles:", os.path.exists(street_poles_file))
print("Police stations:", os.path.exists(police_stations_file))
print("Hospitals:", os.path.exists(hospitals_file))
print("Census:", os.path.exists(census_folder))

File paths defined
Crime files: 5 years
Street poles: True
Police stations: True
Hospitals: True
Census: True


## Explore Crime Data Structure

In [3]:
print("=" * 60)
print("CRIME DATA EXPLORATION")
print("=" * 60)

def explore_crime_file(filepath, year):
    if not os.path.exists(filepath):
        print(year, ": File not found")
        return None
    
    df = pd.read_csv(filepath, nrows=5)
    
    print("\n", year, "CRIME DATA")
    print("Shape (first 5 rows):", df.shape)
    print("\nCOLUMNS (", len(df.columns), "total):")
    for i, col in enumerate(df.columns):
        print(str(i+1) + ". " + col)
    
    print("\nFIRST ROW SAMPLE:")
    first_row = df.iloc[0]
    for col in df.columns[:10]:
        print(col + ":", str(first_row[col])[:50])
    
    return df.columns.tolist()

all_crime_columns = {}
for year, filepath in crime_files.items():
    cols = explore_crime_file(filepath, year)
    if cols:
        all_crime_columns[year] = cols

print("COLUMN CONSISTENCY CHECK:")

if len(all_crime_columns) > 1:
    first_year = list(all_crime_columns.keys())[0]
    base_cols = set(all_crime_columns[first_year])
    
    for year, cols in all_crime_columns.items():
        current_cols = set(cols)
        if base_cols == current_cols:
            print(str(year) + ": Columns match " + str(first_year))
        else:
            print(str(year) + ": Columns differ")
            print("Missing:", base_cols - current_cols)
            print("Extra:", current_cols - base_cols)

CRIME DATA EXPLORATION

 2022 CRIME DATA
Shape (first 5 rows): (5, 19)

COLUMNS ( 19 total):
1. the_geom
2. cartodb_id
3. the_geom_webmercator
4. objectid
5. dc_dist
6. psa
7. dispatch_date_time
8. dispatch_date
9. dispatch_time
10. hour
11. dc_key
12. location_block
13. ucr_general
14. text_general_code
15. point_x
16. point_y
17. gdb_geomattr_data
18. lat
19. lng

FIRST ROW SAMPLE:
the_geom: 0101000020E61000004CC4B3E883CD52C010C86ED1CFFB4340
cartodb_id: 113583
the_geom_webmercator: 0101000020110F0000AA11BA743DF05FC157641625418B5241
objectid: 113563
dc_dist: 16
psa: 1
dispatch_date_time: 2022-07-27 04:00:00+00
dispatch_date: 2022-07-27
dispatch_time: 00:00:00
hour: 13

 2023 CRIME DATA
Shape (first 5 rows): (5, 19)

COLUMNS ( 19 total):
1. the_geom
2. cartodb_id
3. the_geom_webmercator
4. objectid
5. dc_dist
6. psa
7. dispatch_date_time
8. dispatch_date
9. dispatch_time
10. hour
11. dc_key
12. location_block
13. ucr_general
14. text_general_code
15. point_x
16. point_y
17. gdb_geomatt

## Find Latitude/Longitude Columns in Crime Data

In [4]:
print("FINDING COORDINATE COLUMNS IN CRIME DATA:")

sample_crime = pd.read_csv(crime_files[2025], nrows=100)

possible_lat_cols = ['lat', 'latitude', 'y', 'point_y', 'Y', 'LAT', 'Latitude']
possible_lon_cols = ['lng', 'lon', 'longitude', 'x', 'point_x', 'X', 'LON', 'Longitude']

lat_col = None
lon_col = None

print("\nChecking for latitude columns:")
for col in possible_lat_cols:
    if col in sample_crime.columns:
        lat_col = col
        print("Found:", col)
        break
if not lat_col:
    print("No standard latitude column found")

print("\nChecking for longitude columns:")
for col in possible_lon_cols:
    if col in sample_crime.columns:
        lon_col = col
        print("Found:", col)
        break
if not lon_col:
    print("No standard longitude column found")

if lat_col and lon_col:
    print("\nWill use:", lat_col, "and", lon_col, "for coordinates")
    print("\nSample coordinates:")
    sample_coords = sample_crime[[lat_col, lon_col]].head(5)
    print(sample_coords)
else:
    print("\nAll columns in crime data:")
    for col in sample_crime.columns:
        print("-", col)

FINDING COORDINATE COLUMNS IN CRIME DATA:

Checking for latitude columns:
Found: lat

Checking for longitude columns:
Found: lng

Will use: lat and lng for coordinates

Sample coordinates:
         lat        lng
0  39.917257 -75.180239
1  39.912443 -75.242742
2  39.934248 -75.150833
3  39.985319 -75.113792
4  40.050234 -75.094539


## Explore Street Poles

In [5]:
print("STREET POLES DATA:")

if os.path.exists(street_poles_file):
    poles_sample = pd.read_csv(street_poles_file, nrows=5)
    
    print("\nShape (first 5 rows):", poles_sample.shape)
    print("\nCOLUMNS (", len(poles_sample.columns), "total):")
    for i, col in enumerate(poles_sample.columns):
        print("   " + str(i+1) + ". " + col)
    
    print("\nFIRST ROW SAMPLE:")
    for col in poles_sample.columns[:8]:
        print("   " + col + ":", str(poles_sample.iloc[0][col])[:50])
    
    print("\nCOORDINATE CHECK:")
    has_lat = any(col in poles_sample.columns for col in ['lat', 'latitude', 'y', 'Y'])
    has_lon = any(col in poles_sample.columns for col in ['lng', 'lon', 'longitude', 'x', 'X'])
    print("   Has latitude column:", has_lat)
    print("   Has longitude column:", has_lon)
else:
    print("File not found")

STREET POLES DATA:

Shape (first 5 rows): (5, 18)

COLUMNS ( 18 total):
   1. X
   2. Y
   3. oid
   4. pole_num
   5. type
   6. nlumin
   7. lum_size
   8. height
   9. pole_date
   10. up_date
   11. owner
   12. tap_id
   13. block
   14. plate
   15. objectid
   16. light_date
   17. psip_status
   18. bulb_type

FIRST ROW SAMPLE:
   X: -8377994.73428313
   Y: 4848027.49162241
   oid: 0
   pole_num: 100001
   type: SLA
   nlumin: 1.0
   lum_size: 1003.0
   height: 45.0

COORDINATE CHECK:
   Has latitude column: True
   Has longitude column: True


## Explore Police Stations

In [6]:
print("POLICE STATIONS DATA:")

if os.path.exists(police_stations_file):
    police_sample = pd.read_csv(police_stations_file, nrows=5)
    
    print("\nShape (first 5 rows):", police_sample.shape)
    print("\nCOLUMNS (", len(police_sample.columns), "total):")
    for i, col in enumerate(police_sample.columns):
        print("   " + str(i+1) + ". " + col)
    
    print("\nFIRST ROW SAMPLE:")
    for col in police_sample.columns[:8]:
        print("   " + col + ":", str(police_sample.iloc[0][col])[:50])
    
    print("\nCOORDINATE CHECK:")
    lat_cols = [col for col in police_sample.columns if col.lower() in ['lat', 'latitude', 'y']]
    lon_cols = [col for col in police_sample.columns if col.lower() in ['lng', 'lon', 'longitude', 'x']]
    print("   Latitude columns found:", lat_cols)
    print("   Longitude columns found:", lon_cols)
else:
    print("File not found")

POLICE STATIONS DATA:

Shape (first 5 rows): (5, 6)

COLUMNS ( 6 total):
   1. X
   2. Y
   3. objectid
   4. dist_num
   5. location
   6. phone__

FIRST ROW SAMPLE:
   X: -8352601.7108
   Y: 4879170.8476
   objectid: 1
   dist_num: 7
   location: Bustleton Ave & Bowler St
   phone__: 686-3070

COORDINATE CHECK:
   Latitude columns found: ['Y']
   Longitude columns found: ['X']


## Explore Hospitals (GeoJSON)

In [7]:
print("HOSPITALS DATA (GeoJSON):")

if os.path.exists(hospitals_file):
    try:
        hospitals_gdf = gpd.read_file(hospitals_file)
        print("\nShape:", hospitals_gdf.shape)
        print("CRS:", hospitals_gdf.crs)
        
        print("\nCOLUMNS (", len(hospitals_gdf.columns), "total):")
        for i, col in enumerate(hospitals_gdf.columns):
            print("   " + str(i+1) + ". " + col)
        
        print("\nFIRST ROW SAMPLE:")
        for col in hospitals_gdf.columns[:8]:
            val = str(hospitals_gdf.iloc[0][col])
            print("   " + col + ":", val[:50])
    except Exception as e:
        print("Error reading GeoJSON:", e)
else:
    print("File not found")

HOSPITALS DATA (GeoJSON):

Shape: (226, 15)
CRS: EPSG:4326

COLUMNS ( 15 total):
   1. CHIEF_EXEC
   2. OBJECTID
   3. CHIEF_EX_1
   4. FACILITY_U
   5. GEOCODING_
   6. LONGITUDE
   7. COUNTY
   8. FACILITY_N
   9. STREET
   10. FACILITY_I
   11. CITY_OR_BO
   12. LATITUDE
   13. TELEPHONE_
   14. ZIP_CODE
   15. geometry

FIRST ROW SAMPLE:
   CHIEF_EXEC: Chong Park
   OBJECTID: 1
   CHIEF_EX_1: nan
   FACILITY_U: http://www.ahn.org
   GEOCODING_: 00
   LONGITUDE: -80.190773
   COUNTY: Washington
   FACILITY_N: Canonsburg Hospital


## Explore Census Tracts (Shapefile)

In [8]:
print("CENSUS TRACTS DATA (Shapefile):")

import glob

if os.path.exists(census_folder):
    shp_files = glob.glob(census_folder + "/*.shp")
    
    if shp_files:
        census_shp = shp_files[0]
        print("Found shapefile:", census_shp)
        
        try:
            census_gdf = gpd.read_file(census_shp)
            print("\nShape:", census_gdf.shape)
            print("CRS:", census_gdf.crs)
            
            print("\nCOLUMNS (", len(census_gdf.columns), "total):")
            for i, col in enumerate(census_gdf.columns):
                print("   " + str(i+1) + ". " + col)
            
            print("\nFIRST ROW SAMPLE:")
            for col in list(census_gdf.columns)[:8]:
                val = str(census_gdf.iloc[0][col])
                print("   " + col + ":", val[:50])
        except Exception as e:
            print("Error reading shapefile:", e)
    else:
        print("No .shp file found in folder")
else:
    print("Folder not found")

CENSUS TRACTS DATA (Shapefile):
Found shapefile: Data/raw/Census_Tracts_2010-shp\57b9fc6a-ab88-44db-8773-2a46a97483252020328-1-1vm5ejc.1re.shp

Shape: (384, 15)
CRS: EPSG:2272

COLUMNS ( 15 total):
   1. OBJECTID
   2. STATEFP10
   3. COUNTYFP10
   4. TRACTCE10
   5. GEOID10
   6. NAME10
   7. NAMELSAD10
   8. MTFCC10
   9. FUNCSTAT10
   10. ALAND10
   11. AWATER10
   12. INTPTLAT10
   13. INTPTLON10
   14. LOGRECNO
   15. geometry

FIRST ROW SAMPLE:
   OBJECTID: 1
   STATEFP10: 42
   COUNTYFP10: 101
   TRACTCE10: 009400
   GEOID10: 42101009400
   NAME10: 94
   NAMELSAD10: Census Tract 94
   MTFCC10: G5020


## Explore Police Districts

In [9]:
print("POLICE DISTRICTS DATA:")

if os.path.exists(police_districts_folder):
    shp_files = glob.glob(police_districts_folder + "/*.shp")
    
    if shp_files:
        districts_shp = shp_files[0]
        print("Found shapefile:", districts_shp)
        
        try:
            districts_gdf = gpd.read_file(districts_shp)
            print("\nShape:", districts_gdf.shape)
            print("CRS:", districts_gdf.crs)
            
            print("\nCOLUMNS (", len(districts_gdf.columns), "total):")
            for i, col in enumerate(districts_gdf.columns):
                print("   " + str(i+1) + ". " + col)
            
            print("\nFIRST ROW SAMPLE:")
            for col in list(districts_gdf.columns)[:8]:
                val = str(districts_gdf.iloc[0][col])
                print("   " + col + ":", val[:50])
        except Exception as e:
            print("Error reading shapefile:", e)
    else:
        print("No .shp file found in folder")
else:
    print("Folder not found")

POLICE DISTRICTS DATA:
Found shapefile: Data/raw/police_districts\Boundaries_District.shp

Shape: (21, 5)
CRS: EPSG:3857

COLUMNS ( 5 total):
   1. objectid
   2. dist_numc
   3. Shape__Are
   4. Shape__Len
   5. geometry

FIRST ROW SAMPLE:
   objectid: 1
   dist_numc: 24
   Shape__Are: 24312020.33984375
   Shape__Len: 23679.891339329744
   geometry: POLYGON ((-8360704.90490028 4868733.86145551, -836


## Explore Intersection Controls

In [10]:
print("INTERSECTION CONTROLS:")

if os.path.exists(intersection_controls_file):
    intersect_sample = gpd.read_file(intersection_controls_file).head(5)
    
    print("\nShape (first 5 rows):", intersect_sample.shape)
    print("\nCOLUMNS (", len(intersect_sample.columns), "total):")
    for i, col in enumerate(intersect_sample.columns):
        print("   " + str(i+1) + ". " + col)
    
    print("\nFIRST ROW SAMPLE:")
    for col in intersect_sample.columns[:8]:
        print("   " + col + ":", str(intersect_sample.iloc[0][col])[:50])
    
    print("\nCOORDINATE CHECK:")
    has_coords = any(col in intersect_sample.columns for col in ['lat', 'lng', 'lon', 'x', 'y'])
    print("   Has coordinate columns:", has_coords)
else:
    print("File not found")

INTERSECTION CONTROLS:

Shape (first 5 rows): (5, 5)

COLUMNS ( 5 total):
   1. objectid
   2. stoptype
   3. node_id
   4. led_status
   5. geometry

FIRST ROW SAMPLE:
   objectid: 1
   stoptype: Conventional
   node_id: 2
   led_status: nan
   geometry: POINT (-75.0145624057219 40.1376887660274)

COORDINATE CHECK:
   Has coordinate columns: False


## Summary of All Data

In [11]:
summary = []

crime_years_available = []
for year, filepath in crime_files.items():
    if os.path.exists(filepath):
        crime_years_available.append(year)
summary.append("Crime data: " + str(len(crime_years_available)) + " years (" + str(crime_years_available) + ")")

files_to_check = [
    ("Street Poles", street_poles_file),
    ("Police Stations", police_stations_file),
    ("Hospitals", hospitals_file),
    ("Census Tracts", census_folder),
    ("Police Districts", police_districts_folder),
    ("Intersection Controls", intersection_controls_file),
    ("Weather", weather_file),
]

for name, path in files_to_check:
    exists = os.path.exists(path)
    if name == "Weather" and exists:
        try:
            weather_df = pd.read_csv(path, nrows=2)
            summary.append(name + ": Yes (" + str(weather_df.shape[1]) + " columns)")
        except:
            summary.append(name + ": Yes but check format")
    else:
        status = "Yes" if exists else "No"
        summary.append(name + ": " + status)

print("\nAvailable datasets:")
for item in summary:
    print("   " + item)


Available datasets:
   Crime data: 5 years ([2022, 2023, 2024, 2025, 2026])
   Street Poles: Yes
   Police Stations: Yes
   Hospitals: Yes
   Census Tracts: Yes
   Police Districts: Yes
   Intersection Controls: Yes
   Weather: Yes (26 columns)
